In [ ]:
Imports

In [ ]:
import tensorflow as tf
from keras._tf_keras.keras.models import Model
from keras._tf_keras.keras.layers import (Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout, TimeDistributed, BatchNormalization, Input, Reshape)
from keras._tf_keras.keras.preprocessing.image import ImageDataGenerator

import matplotlib.pyplot as plt



In [ ]:
# Input shape (adjust as per your dataset)
input_shape = (128, 128, 1)  # Example for grayscale signatures

# CNN Feature Extractor
cnn_input = Input(shape=input_shape)
x = Conv2D(32, (3,3), activation='relu', padding='same')(cnn_input)
x = MaxPooling2D((2,2))(x)
x = BatchNormalization()(x)

x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = BatchNormalization()(x)

x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = BatchNormalization()(x)

x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

# Reshape for RNN (assuming a sequence length of 16)
x = Reshape((16, 16))(x)

# RNN Layer
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64)(x)

# Fully Connected Output Layer
output = Dense(1, activation='sigmoid')(x)

# Define Model
model = Model(inputs=cnn_input, outputs=output)

# Compile Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary
model.summary()

In [ ]:

# Define data directories
train_dir = "./Signature-Forgery-Detection-System/Dataset/Dataset_Split/Train"
test_dir = "./Signature-Forgery-Detection-System/Dataset/Dataset_Split/Test"

# ImageDataGenerator for training (with augmentation) and testing
train_datagen = ImageDataGenerator(rescale=1./255, 
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True,
                                   fill_mode='nearest')

test_datagen = ImageDataGenerator(rescale=1./255)

# Load the data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    color_mode='grayscale',
    batch_size=32,
    class_mode='binary',  # since you're detecting genuine/forgery
    shuffle=True)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    color_mode='grayscale',
    batch_size=32,
    class_mode='binary',
    shuffle=False)

Training the Model

In [ ]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=20,
    validation_data=test_generator,
    validation_steps=test_generator.samples // test_generator.batch_size)


Model Evaluation

In [ ]:
# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

Save the Model

In [ ]:
# Save the trained model
model.save('HYBRID(CNN+RNN)_signature_forgery_model.keras')


 Visualization of Training and Validation Metrics

In [ ]:
# Plot training & validation accuracy
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot training & validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()